# Dynamic Protein Modeling — Membrane Proteins & Fine-Tuning Applications
## GPCRs, Ion Channels, Lipid Bilayer Embeddings, Conformational Dynamics

**TL;DR (Plain English):** Membrane proteins are embedded in lipid bilayers and act as molecular gates, sensors, and pumps — ~70% of approved drug targets are membrane proteins. They are notoriously hard to study experimentally (hard to crystallize) and computationally (require lipid environment, have multiple conformations). This notebook covers:
1. **What makes membrane proteins special** (transmembrane helices, lipid interactions, conformational states)
2. **Fine-tuning protein language models for membrane proteins** (domain adaptation from soluble → membrane)
3. **Molecular dynamics (MD) concepts** — how to model protein motion, not just static structure
4. **Deep learning for conformational dynamics** — learning the energy landscape
5. **GPCR structure prediction** — the most important drug target class

**After this notebook you can:**
- Fine-tune ESM2 specifically for membrane protein sequences
- Predict transmembrane topology (inside-out orientation, helix positions)
- Understand why AF3 struggles with membrane proteins and how to fix it
- Build a conformational ensemble predictor using latent diffusion
- Connect to drug discovery: GPCR docking, ion channel blockers

## Beginner Teaching Frame

**Who should fully work through this notebook:** advanced students. Most learners should use a concept-first first pass.

**How to study it on a first pass:**
- focus on why membrane proteins require different modeling assumptions
- connect topology, lipid environment, and conformational state to the ML choices
- treat this as an elective, not a prerequisite for core competence

**Common traps:** trying to master every modeling detail before you can explain the biology problem.


## Canonical Resource Upgrade

Helpful references:
- [OPM](https://opm.phar.umich.edu/)
- [GPCRdb](https://gpcrdb.org/)
- [DeepTMHMM](https://dtu.biolib.com/DeepTMHMM)


## Prerequisites & Learning Path

| | |
|--|--|
| Prerequisites | 10/01 Fine-tuning Capstone (LoRA, delta-delta-G), 07/01 AF3 Architecture, 03/01 Structural Biology |
| You Are Here | Module 11/01 — Membrane Protein Dynamics (advanced application) |
| Next Steps | Original research / industry role — this is the frontier |
| Goal | Fine-tune protein models for membrane proteins; understand conformational dynamics |

### From Scratch? Start Here:
1. **Step 1:** [Membrane proteins intro — iBiology (free)](https://www.ibiology.org/cell-biology/membrane-proteins/) — 1 hour
2. **Step 2:** [GPCR structure — PDB-101 tutorial](https://pdb101.rcsb.org/motm/227) — visual intro
3. **Step 3:** Complete 10/01 (fine-tuning capstone) — LoRA, Pairformer, delta-delta-G
4. **Step 4:** This notebook

**Time:** 20-30 hours | **Difficulty:** Research-level

### Cross-References
- Builds on: 10/01 (LoRA fine-tuning), 07/01-04 (AF3), 06/02 (GNNs for structure), 03/01 (PDB, RMSD)
- Used by: Original research / industry applications
- Related: 10/01 covers general delta-delta-G fine-tuning; this notebook applies it specifically to membrane proteins with conformational dynamics

## Section 1 — Membrane Protein Biology

### What Are Membrane Proteins?

Membrane proteins are embedded in or associated with the lipid bilayer — the 30-angstrom hydrophobic slab separating the inside and outside of every cell. They account for ~30% of the human proteome but ~60-70% of drug targets because they control:

- **Signal transduction** — GPCRs detect hormones, neurotransmitters, odors, light, and transduce signal into the cell
- **Ion transport** — voltage-gated Na+/K+/Ca2+ channels control nerve firing, heartbeat, muscle contraction
- **Metabolite transport** — ABC transporters, SLC transporters move nutrients, drugs, toxins across membranes
- **Cell adhesion** — integrins, cadherins hold cells together in tissues

### The Hydrophobic Core Problem

The lipid bilayer core is highly hydrophobic (~16 kcal/mol free energy cost to bury a charged residue). Transmembrane helices solve this by:
1. Using hydrophobic residues (Ile, Leu, Val, Phe, Met) facing the lipid
2. Satisfying backbone H-bonds within the helix (not to water)
3. Maintaining ~20 residues per helix to span the ~30-angstrom bilayer

### Key Classes

| Class | TM Helices | Examples | Drug Targets |
|-------|------------|---------|------|
| GPCRs | 7 | Beta-2 adrenergic, Rhodopsin, mu-opioid | ~35% all approved drugs |
| Ion channels | 2-6 | VGSC, VGKC, nAChR | Anticonvulsants, anesthetics |
| Transporters | 12+ | P-glycoprotein, SERT, DAT | Antidepressants, cancer drugs |
| Receptor tyrosine kinases | 1 | EGFR, HER2, VEGFR | Cancer drugs (gefitinib, trastuzumab) |
| Beta-barrels | 8-22 | OmpF, BamA | Antibiotic targets |

**Key insight for ML:** These classes have different TM topology, different sequence statistics, and require different fine-tuning strategies.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Membrane protein biology — key facts + topology prediction
print("Membrane Protein Classification")
print("=" * 50)

classes = {
    "Type I single-pass": "N-out, C-in (e.g. CD4, growth factor receptors)",
    "Type II single-pass": "N-in, C-out (e.g. transferrin receptor)",
    "Multi-pass (GPCR-like)": "7 TM helices, N-out, C-in (e.g. rhodopsin)",
    "Multi-pass (ion channel)": "4-6 TM subunits (e.g. potassium channel)",
    "Beta-barrel (OMP)": "Gram-negative outer membrane (e.g. porins)",
}
for cls, example in classes.items():
    print(f"  {cls}: {example}")

print()
# Transmembrane helix hydrophobicity scan
amino_acid_hydrophobicity = {
    'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,'Q':-3.5,'E':-3.5,'G':-0.4,
    'H':-3.2,'I':4.5,'L':3.8,'K':-3.9,'M':1.9,'F':2.8,'P':-1.6,'S':-0.8,
    'T':-0.7,'W':-0.9,'Y':-1.3,'V':4.2
}

def kyte_doolittle(sequence, window=19):
    """Kyte-Doolittle hydrophobicity scan for TM helix prediction."""
    scores = []
    for i in range(len(sequence) - window + 1):
        window_seq = sequence[i:i+window]
        score = np.mean([amino_acid_hydrophobicity.get(aa, 0) for aa in window_seq])
        scores.append(score)
    return scores

# Bacteriorhodopsin-like sequence (7 TM helices)
seq = "MLELLPTAVEGVSQAQITGRPEWIWLALGTALMGLGTLYFLVKGMGVSDPDAKKFYAITTLVPAIAFTMYLSMLLGYGLTMVPFGGEQNPIYWARYADWLFTTPLLLLDLALLVDADEQGTILALVGADGIMIGTGLVGALTKVYSYRFVWWAISTAAMLYILYVLFFGFTSKAESMRPEVASTFKVLRNVTVVLWSAYPVVWLIGSEGAGIVPLNIETLLFMVLDVSAKVGFGLILLRSRAI"
scores = kyte_doolittle(seq[:100])
print(f"KD scan (first 82 positions):")
print(f"  Max score: {max(scores):.2f} at position {np.argmax(scores)+1}")
print(f"  TM helix candidates (score > 1.6): {sum(s > 1.6 for s in scores)}")
print(f"  TM helix prediction threshold: 1.6 (Kyte-Doolittle scale)")

## Section 2 — Why AF3 Struggles with Membrane Proteins

### The Training Data Problem

AlphaFold3 was trained primarily on experimentally determined structures from the PDB. This creates a severe bias:

- **PDB composition:** ~10% of structures are membrane proteins
- **Human proteome:** ~30% of genes encode membrane proteins
- **Drug targets:** ~60-70% are membrane proteins

This 3-10x under-representation means AF3's internal statistics (attention weights, pair representations, structure module biases) are dominated by soluble protein geometry.

### Why Crystallography Fails for Membrane Proteins

To get a crystal structure you need a crystal. Membrane proteins:
1. Require detergent micelles or nanodiscs to stay soluble outside the membrane
2. Are conformationally flexible — hard to trap in a single state
3. Have disordered loops (ICL3 in GPCRs) that prevent crystal packing
4. Often need binding partners (G-proteins, arrestins) to stabilize active state

Result: 10x fewer structures than soluble proteins of similar importance.

### AF3-Specific Failures

| Issue | Why It Matters | Fix |
|-------|---------------|-----|
| No lipid bilayer context | TM helices tilt and pack differently in lipid vs vacuum | Include lipid beads in input (Chai-1 approach) |
| Single conformational state | GPCRs have active + inactive states differing by 10+ angstroms | Conditional generation or ensemble prediction |
| ICL3 loop disordered | Loop that couples to G-protein is often wrong | Domain-specific fine-tuning |
| MSA depth lower | Fewer homologs for rare TM proteins | Augment with structure-based alignment |
| TM helix registry errors | Helices shift by 3-4 residues = completely wrong structure | LoRA fine-tuning on OPM structures |

In [ ]:
import torch
import torch.nn as nn

# Why AF3 struggles with membrane proteins + domain adaptation
print("AF3 vs Membrane Proteins: The Training Data Problem")
print("=" * 60)
print()
print("PDB composition (as of 2024):")
pdb_stats = {
    "Soluble proteins": "~85%",
    "Membrane proteins": "~3-5%",
    "Protein-DNA complexes": "~5%",
    "Protein-RNA complexes": "~3%",
    "Other": "~2-4%",
}
for cat, pct in pdb_stats.items():
    print(f"  {cat}: {pct}")

print()
print("Why membrane proteins are underrepresented:")
reasons = [
    "Crystallography requires detergent micelles or lipid cubic phases",
    "NMR limited to small membrane proteins",
    "Cryo-EM improving but still challenging for small MPs",
    "Expression and purification much harder than soluble proteins",
]
for r in reasons:
    print(f"  • {r}")

print()
print("Consequence for AF3:")
consequences = [
    "Less training data → lower confidence (pLDDT) for TM regions",
    "Lipid environment not modeled → may mispredict helix tilt",
    "Detergent artifacts in training structures",
    "PAE may be unreliable for inter-helix packing",
]
for c in consequences:
    print(f"  ! {c}")

# Simple domain adaptation: freeze backbone, train with membrane-specific data
class MembraneAdaptedESM(nn.Module):
    def __init__(self, esm_dim=320, n_freeze_layers=30):
        super().__init__()
        # Simulated ESM2 backbone (freeze first n_freeze_layers)
        self.backbone = nn.Sequential(*[
            nn.Linear(esm_dim, esm_dim) for _ in range(33)
        ])
        for i, layer in enumerate(self.backbone):
            layer.weight.requires_grad = (i >= n_freeze_layers)

        # Membrane-specific head
        self.tm_predictor = nn.Sequential(
            nn.Linear(esm_dim, 128), nn.ReLU(),
            nn.Linear(128, 2)  # TM vs non-TM
        )

    def forward(self, x):
        h = x
        for layer in self.backbone:
            h = torch.relu(layer(h))
        return self.tm_predictor(h)

model = MembraneAdaptedESM()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nDomain-adapted ESM: {trainable:,}/{total:,} params trainable ({trainable/total:.1%})")

## Section 3 — Fine-Tuning ESM2 for Membrane Proteins

### Domain Adaptation Strategy

ESM2 was trained on 250M protein sequences from UniRef90. It knows the language of proteins but has seen far more soluble proteins than membrane proteins.

**Domain adaptation pipeline:**
1. Start from `facebook/esm2_t6_8M_UR50D` (or larger variant)
2. Apply LoRA adapters to Q/K/V attention layers (r=16, alpha=32)
3. Fine-tune on OPM membrane protein sequences with topology labels
4. Freeze the main ESM2 weights — only LoRA + task head are trainable
5. Evaluate on held-out membrane proteins from MPstruc

### Why LoRA Works Here

LoRA decomposes the weight update as W_new = W_pretrained + (alpha/r) * B * A:
- **W_pretrained** (frozen): captures universal protein language (~8M parameters for ESM2-8M)
- **B * A** (trainable): captures membrane-specific corrections (~150k parameters for r=16)
- **Ratio:** 1.8% trainable, 98.2% frozen — efficient domain adaptation

The key insight: membrane proteins use the same amino acid vocabulary as soluble proteins. ESM2's embeddings are a good starting point — we just need to adjust the attention patterns for TM-specific residue co-evolution patterns.

### Topology Labels

The training labels come from OPM (Orientations of Proteins in Membranes):
- **Label 0 (Inside):** Cytoplasmic residues
- **Label 1 (TM):** Transmembrane residues
- **Label 2 (Outside):** Extracellular / periplasmic residues

This is a sequence labeling task (like NER in NLP) — every residue gets a topology label.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# ESM2 LoRA fine-tuning for membrane protein TM topology
class ESM2LoRA(nn.Module):
    """Simulate ESM2 fine-tuning with LoRA for TM helix prediction."""
    def __init__(self, d_model=320, n_heads=20, rank=8, n_classes=2):
        super().__init__()
        # Frozen backbone simulation
        self.embed = nn.Embedding(21, d_model)
        self.embed.weight.requires_grad = False
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        for p in self.attn.parameters():
            p.requires_grad = False

        # LoRA on query and value projections
        self.lora_q_A = nn.Parameter(torch.randn(rank, d_model) * 0.01)
        self.lora_q_B = nn.Parameter(torch.zeros(d_model, rank))
        self.lora_v_A = nn.Parameter(torch.randn(rank, d_model) * 0.01)
        self.lora_v_B = nn.Parameter(torch.zeros(d_model, rank))
        self.scaling = 16 / rank  # alpha/rank

        # TM prediction head (trainable)
        self.tm_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(64, n_classes)
        )

    def forward(self, tokens):
        x = self.embed(tokens)
        # Standard attention + LoRA correction
        q_correction = (x @ self.lora_q_A.T) @ self.lora_q_B.T * self.scaling
        v_correction = (x @ self.lora_v_A.T) @ self.lora_v_B.T * self.scaling
        h, _ = self.attn(x + q_correction, x, x + v_correction)
        return self.tm_head(h)  # (B, L, 2) per-residue TM/non-TM

torch.manual_seed(42)
model = ESM2LoRA(d_model=64, n_heads=4, rank=4)  # small for demo
B, L = 4, 50
tokens = torch.randint(0, 21, (B, L))
logits = model(tokens)
print(f"TM prediction: {logits.shape}  (batch, seq_len, classes)")
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,}/{total:,} ({trainable/total:.1%})")
print("Output: per-residue TM (1) vs non-TM (0) prediction")

## Section 4 — Conformational Dynamics with ML

### The Conformational Landscape Problem

Proteins are not static structures — they are dynamic ensembles. A single PDB file captures one snapshot. For membrane proteins, this is especially limiting:

- **GPCRs** exist in inactive state (no ligand), intermediate states, and fully active state (G-protein bound)
- **Ion channels** cycle through closed, open, and inactivated states on millisecond timescales
- **Transporters** use alternating access: inward-facing and outward-facing conformations

The energy landscape (free energy as a function of conformation) determines which states are populated. Drug discovery needs to know:
1. Which conformation does my drug bind to?
2. Does my drug shift the equilibrium toward active or inactive state?
3. What is the binding free energy (delta-G) in each state?

### ML Approaches to Conformational Dynamics

| Approach | Key Idea | Reference |
|---------|----------|----------|
| VAE on structure features | Encode structures into latent space; interpolate between states | AlphaFlow (2024) |
| Diffusion on coordinates | Generate MD-like trajectories from sequence | MDGen (Jing 2024) |
| Markov State Models + ML | Cluster MD trajectories; model transitions with ML | PyEMMA |
| Neural ODE | Learn continuous trajectory in latent space | Various (2022-2024) |
| Boltzmann generator | Generate equilibrium samples from target distribution | Noe group (2019+) |

### VAE Architecture for Conformational States

We use a Variational Autoencoder to learn a structured latent space where:
- Active GPCR structures cluster in one region
- Inactive GPCR structures cluster in another region
- The path between them corresponds to the activation pathway

The VAE has three loss terms:
1. **Reconstruction loss** (MSE): can we recover the input structure features from z?
2. **KL divergence**: keep the latent space smooth and continuous
3. **Classification loss** (cross-entropy): make the latent space discriminative for state

In [ ]:
import numpy as np
import torch

# Conformational dynamics — GPCR activation simulation
print("Conformational Dynamics: GPCR Activation Example")
print("=" * 60)

# GPCR states: inactive (R), intermediate (R*), active (R**)
# Simplified 2D conformational landscape (TMH6 outward movement)
np.random.seed(42)

def free_energy_landscape(x, y, state='inactive'):
    """Simplified GPCR free energy landscape.
    x: TM6 outward movement (Å)
    y: TM3/TM7 rotation angle
    """
    if state == 'inactive':
        # Inactive: minimum at (0, 0)
        return 0.5*x**2 + 0.5*y**2
    elif state == 'active':
        # Active: minimum at (6, 30) — TM6 moves out ~6Å, rotation ~30°
        return 0.5*(x-6)**2 + 0.2*(y-30)**2

# Monte Carlo sampling of conformations
def mc_sample(landscape_fn, n_steps=1000, T=1.0):
    x, y = 0.0, 0.0
    traj = [(x, y)]
    E = landscape_fn(x, y)
    for _ in range(n_steps):
        dx = np.random.normal(0, 0.5)
        dy = np.random.normal(0, 2.0)
        E_new = landscape_fn(x+dx, y+dy)
        if E_new < E or np.random.rand() < np.exp(-(E_new-E)/T):
            x, y, E = x+dx, y+dy, E_new
        traj.append((x, y))
    return np.array(traj)

# Sample inactive and active states
traj_inactive = mc_sample(lambda x,y: free_energy_landscape(x,y,'inactive'))
traj_active   = mc_sample(lambda x,y: free_energy_landscape(x,y,'active'))

print(f"Inactive state: TM6 shift = {np.mean(traj_inactive[:,0]):.2f} ± {np.std(traj_inactive[:,0]):.2f} Å")
print(f"Active state:   TM6 shift = {np.mean(traj_active[:,0]):.2f} ± {np.std(traj_active[:,0]):.2f} Å")
print(f"Activation: TM6 moves outward ~6 Å (observed in β2AR crystal structures)")
print()
print("ML approaches to conformational dynamics:")
print("  1. Variational autoencoder (VAE) on MD trajectories")
print("  2. Deep TICAe / time-lagged autoencoders for slow modes")
print("  3. Graph neural networks on coarse-grained conformations")
print("  4. AlphaFold2 with modified templates for alternative conformations")

## Section 5 — Coarse-Grained MD + ML Hybrid

### Why Coarse-Graining?

All-atom MD simulation of a GPCR in a lipid bilayer:
- System size: ~100,000 atoms (protein + lipids + water + ions)
- Timestep: 2 femtoseconds (limited by fastest bond vibration)
- GPCR activation timescale: ~1 microsecond
- Required steps: 500,000,000 (500M steps)
- Cost: ~1 week on 4 GPUs (or 1 day on Anton2 specialized hardware)

**Coarse-graining** maps multiple atoms to single interaction sites ("beads"):
- MARTINI force field: ~4 heavy atoms per bead
- Protein: 1-2 beads per residue (backbone + sidechain)
- Lipid: ~10 beads per molecule
- Speed gain: 100-1000x faster, enables microsecond timescales on standard GPUs

### ML-Enhanced Coarse-Grained MD

Traditional CG force fields (MARTINI) are hand-designed. ML can learn them from data:

1. **Neural network potentials (NNPs):** Learn energy function E(coordinates) from high-level QM or AA-MD data
2. **Equivariant GNNs:** Architecture must be rotation/translation invariant — use SE(3)-equivariant layers (EGNN, NequIP, MACE)
3. **Force matching:** Train on forces from AA-MD trajectories using L2 loss
4. **Boltzmann generators:** Train normalizing flow to directly sample equilibrium distribution

### EGNN Architecture for CG Potentials

Equivariant Graph Neural Networks update both node features (h) AND coordinates (x):
- Node message: m_ij = phi_e(h_i, h_j, ||x_i - x_j||^2) — rotation invariant (uses squared distances)
- Coordinate update: x_i += sum_j (x_i - x_j) * phi_x(m_ij) — equivariant (scales displacement vectors)
- Feature update: h_i = phi_h(h_i, sum_j m_ij) — standard message passing

This ensures: rotate all input coordinates by R -> output coordinates rotate by R too. Critical for physical correctness.

In [ ]:
import numpy as np

# Coarse-grained MD + ML hybrid (MARTINI CG)
print("MARTINI Coarse-Grained MD + ML")
print("=" * 60)
print()
print("MARTINI force field:")
print("  ~4 heavy atoms → 1 bead (backbone + sidechain beads)")
print("  DPPC lipid: 14 beads (vs ~128 atoms all-atom)")
print("  GPCR: ~200 beads (vs ~3500 atoms)")
print("  ~100× speedup over all-atom MD")
print()

# Simulate CG bead assignment for a helix
aa_to_beads = {
    'ALA': 2, 'GLY': 1, 'VAL': 2, 'LEU': 3, 'ILE': 3,
    'PHE': 4, 'TRP': 5, 'TYR': 4, 'MET': 3, 'CYS': 2,
    'SER': 2, 'THR': 2, 'ASN': 3, 'GLN': 3, 'ASP': 3,
    'GLU': 3, 'LYS': 4, 'ARG': 5, 'HIS': 4, 'PRO': 2,
}
aa_code = {'A':'ALA','G':'GLY','V':'VAL','L':'LEU','I':'ILE',
           'F':'PHE','W':'TRP','Y':'TYR','M':'MET','C':'CYS',
           'S':'SER','T':'THR','N':'ASN','Q':'GLN','D':'ASP',
           'E':'GLU','K':'LYS','R':'ARG','H':'HIS','P':'PRO'}

helix_seq = "AAWILLSSQLSTAFGIIAAD"
total_beads = sum(aa_to_beads.get(aa_code.get(aa, 'ALA'), 2) for aa in helix_seq)
total_atoms = len(helix_seq) * 7  # avg ~7 heavy atoms/residue
print(f"Helix ({len(helix_seq)} residues): {total_atoms} atoms → {total_beads} CG beads")
print(f"Compression: {total_atoms/total_beads:.1f}× fewer degrees of freedom")

print()
print("ML + CG hybrid approaches:")
approaches = [
    "BackMapping: CG → all-atom reconstruction via ML",
    "CG force field learning: match all-atom PMF with ML potential",
    "Enhanced sampling: ML-guided collective variables for REMD",
    "CGMD + GNN: learn CG force field directly from QM data",
]
for a in approaches:
    print(f"  • {a}")

## Section 6 — GPCR Drug Discovery Application

### Why GPCRs Dominate Drug Discovery

GPCRs (G protein-coupled receptors) are 7-TM helical proteins that:
1. Detect extracellular signals (hormones, neurotransmitters, sensory inputs)
2. Transduce signals through G-proteins (Gs, Gi, Gq, G12/13 subtypes) OR beta-arrestin
3. Control essentially every major physiological process

**The statistics:**
- ~800 human GPCRs encoded in the genome
- ~35% of all FDA-approved drugs target GPCRs
- ~$200 billion annual GPCR drug market
- Drugs for: heart disease, asthma, schizophrenia, depression, pain, diabetes (Ozempic!), migraines, hypertension...

### The Active vs Inactive State Problem

This is the central challenge for computational GPCR drug discovery:

- **Inactive state:** Receptor sits in lipid bilayer, no G-protein bound, lower affinity for agonists
- **Active state:** Agonist bound, G-protein coupled, TM6 moves 14 angstroms outward, completely different binding pocket geometry
- **Biased agonism:** Some drugs activate G-protein signaling but not beta-arrestin (or vice versa) — potentially better safety profiles

**For ML:** You need to know WHICH state to dock your drug to. Docking an agonist to the inactive state structure gives wrong affinity predictions.

### The Computational Workflow

```
Sequence --(ESM2+LoRA)--> Embedding
                                 |
                    Boltz-2/AF3 structure prediction
                                 |
                  Active & Inactive state ensembles
                                 |
               Virtual screening: dock 10k-1M compounds
                                 |
              Rank by predicted binding affinity (dG)
                                 |
             Top 100 hits -> FEP+ for high-accuracy dG
                                 |
                  Experimental validation (IC50, Kd)
```

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# GPCR drug discovery application
print("GPCR Drug Discovery: Structure-Based Design")
print("=" * 60)
print()
print("GPCR facts:")
print("  >800 human GPCRs; ~35% of all drug targets")
print("  All signal through G-proteins (Gs, Gi, Gq) or β-arrestin")
print("  Key challenge: selectivity (many similar binding pockets)")
print()

# Simulate virtual screening score
class GPCRBindingScorer(nn.Module):
    """Simple binding affinity predictor from protein+ligand features."""
    def __init__(self, protein_dim=128, ligand_dim=64):
        super().__init__()
        self.protein_enc = nn.Sequential(nn.Linear(protein_dim, 64), nn.ReLU())
        self.ligand_enc  = nn.Sequential(nn.Linear(ligand_dim,  64), nn.ReLU())
        self.interaction = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1)  # predicted pKi
        )

    def forward(self, p_feat, l_feat):
        p = self.protein_enc(p_feat)
        l = self.ligand_enc(l_feat)
        return self.interaction(torch.cat([p, l], dim=-1))

torch.manual_seed(42)
model = GPCRBindingScorer()
# Simulate batch of 10 protein-ligand pairs
p_feat = torch.randn(10, 128)  # binding pocket features
l_feat = torch.randn(10, 64)   # ligand Morgan fingerprint
scores = model(p_feat, l_feat).squeeze()
print(f"Predicted pKi values: {scores.detach().numpy().round(2)}")
print(f"pKi > 9 = sub-nM affinity (very potent)")
print()
print("State-of-the-art GPCR ML tools:")
tools = ["DiffDock: diffusion-based docking", "Glide/Vina: physics-based docking",
         "AlphaFold3: complex structure prediction", "SchNet/DimeNet: molecular ML potentials",
         "REINVENT: RL-based ligand generation"]
for t in tools:
    print(f"  • {t}")

## Resources

### 1. Theory — Foundations and Math
- **Membrane biophysics:** Fluid mosaic model, lipid bilayer thermodynamics, hydrophobic effect (~4 kJ/mol per CH2 transferred to water)
- **Transmembrane helix physics:** Hydrophobicity scales (Kyte-Doolittle, White-Wimley), helix-coil theory, translocon biology
- **GPCR activation mechanism:** Molecular switch theory, G-protein coupling stoichiometry, beta-arrestin biased signaling
- **Molecular dynamics theory:** Langevin equation, MARTINI force field, free energy perturbation (FEP), enhanced sampling (metadynamics, replica exchange)
- **ML force fields:** Neural network potentials (NNP) theory, equivariance requirements, ANI/DimeNet++/NequIP
- Papers: [AlphaFold for membrane proteins (2023)](https://www.nature.com/articles/s41467-023-36943-y), [MARTINI 3 (2021)](https://www.nature.com/articles/s41592-021-01098-3), [MDGen (Jing 2024)](https://arxiv.org/abs/2402.04594)

### 2. Must-Have Popular Resources
- Book: Molecular Biology of the Cell (Alberts) — ch. 10-11, membrane proteins and cell signaling
- Book: Biological Physics (Nelson) — membrane biophysics quantitatively
- Course: [iBiology Membrane Proteins](https://www.ibiology.org/cell-biology/membrane-proteins/) — free, expert video lectures
- Course: [GPCRdb tutorials](https://gpcrdb.org/docs/reference/generic_numbering) — GPCR-specific database and analysis
- GitHub: [OpenMM/openmm](https://github.com/openmm/openmm) — GPU-accelerated MD simulation
- GitHub: [markovmodel/PyEMMA](https://github.com/markovmodel/PyEMMA) — Markov state models for protein dynamics
- GitHub: [dptech-corp/Uni-Mol](https://github.com/dptech-corp/Uni-Mol) — molecular pretraining with 3D structure
- HuggingFace: [facebook/esmfold_v1](https://huggingface.co/facebook/esmfold_v1) — structure prediction starting point
- Kaggle: [Protein-Ligand Interaction datasets](https://www.kaggle.com/search?q=protein+ligand+interaction)

### 3. Practicals — Hands-On
- Exercise 1: Predict TM topology for 10 GPCRs using KD hydrophobicity + compare to [Phobius](https://phobius.sbc.su.se/)
- Exercise 2: Download an active + inactive GPCR pair from PDB (beta2-adrenergic: 3SN6 active, 2RH1 inactive), compute RMSD and visualize the conformational change
- Exercise 3: Fine-tune ESM2 (use HuggingFace PEFT) on membrane protein sequences from OPM database
- Exercise 4: Implement a simple Markov State Model on a toy MD trajectory to identify metastable states
- GitHub: [MDAnalysis/MDAnalysis](https://github.com/MDAnalysis/MDAnalysis) — structure and trajectory analysis
- GitHub: [chaitjo/geometric-gnn-dojo](https://github.com/chaitjo/geometric-gnn-dojo) — equivariant GNNs for proteins
- GitHub: [microsoft/AI2BMD](https://github.com/microsoft/AI2BMD) — Microsoft AI-enhanced MD
- HuggingFace: [OPM dataset](https://opm.phar.umich.edu/) — download GPCR structures with membrane orientation
- Kaggle: [GPCRmd molecular dynamics database](https://www.gpcrmd.org/) — ready-to-use MD trajectories

### 4. Real-World Problems
- AstraZeneca: Uses AF3 + MD for GPCR drug discovery — combining structure prediction with dynamics
- D.E. Shaw Research (DESRES): Anton supercomputer, millisecond MD for GPCR activation
- Schrodinger: FEP+ for membrane protein binding affinity, used in production drug design
- Exscientia: AI-designed drugs for GPCRs (first AI-designed GPCR drug in Phase I)
- Dataset: [GPCRmd](https://www.gpcrmd.org/) — 160+ GPCR MD trajectories (free, open access)
- Dataset: [OPM database](https://opm.phar.umich.edu/) — 9,000 membrane protein structures with orientation
- Dataset: [MPstruc](https://blanco.biomol.uci.edu/mpstruc/) — 900+ unique membrane protein families
- HuggingFace: [GPCR-Bench dataset](https://huggingface.co/datasets) — binding affinity measurements
- Application: Predict whether a GPCR mutation (disease variant) activates or inactivates the receptor — connects to delta-delta-G prediction from Module 10

### 5. Interview Tips
- Must know: What percentage of approved drugs target membrane proteins? (~60%) Which class? (GPCRs ~35%)
- Must know: Why is structure prediction harder for membrane proteins than soluble proteins? (under-represented in PDB, multiple conformational states, lipid environment)
- Must know: What is the MARTINI force field and why is coarse-graining useful for membrane proteins?
- Must know: What are the 4 G-alpha protein families (Gs, Gi, Gq, G12) and how does biased agonism exploit them?
- Common mistake: Assuming AF3 handles membrane proteins well — it significantly under-performs vs soluble proteins (limited membrane protein training data)
- Common mistake: Forgetting that GPCRs have at least 2 major conformational states (active/inactive) — a single static prediction misses the biology
- Common mistake: Using regular MD without enhanced sampling for slow processes (GPCR activation takes ~1 microsecond, standard MD runs ~ns)
- Pro tip: At computational biology ML teams/AstraZeneca interviews, demonstrate you know the connection between GPCR conformational state and signaling outcome (same receptor -> different drugs -> different states -> different biology)
- Pro tip: Know GPCRdb (gpcrdb.org) — the GPCR expert's go-to database, used by every computational chemist in the field

### 6. Hot Industry Topics
- Trending: [MDGen (Jing 2024)](https://arxiv.org/abs/2402.04594) — diffusion model generating MD trajectories from sequence alone
- Trending: [ESM3](https://huggingface.co/EvolutionaryScale/esm3-sm-open-v1) — multimodal protein model handles structure + sequence + function, better for membrane proteins
- Trending: [Boltz-2](https://github.com/jwohlwend/boltz) — permissive license AF3 alternative, actively fine-tuned for membrane proteins by academic groups
- Trending: [NequIP/MACE](https://github.com/ACEsuit/mace) — equivariant ML force fields replacing classical MD for protein/membrane systems
- Trending: [AlphaFold Multimer + membrane](https://github.com/google-deepmind/alphafold) — AF3 extensions for membrane-embedded complex prediction
- Trending: [GPCRfold](https://github.com/spyakub/GPCRfold) — AF3 specialized for GPCR conformational states
- Build: Fine-tune ESM2 on GPCRdb sequences (LoRA, r=16), evaluate TM topology prediction on held-out GPCRs
- Build: Train a conformational state classifier (active/inactive) on GPCR embeddings from GPCRmd trajectories
- Build: Markov State Model pipeline: GPCRmd trajectory -> feature extraction -> MSM -> identify metastable states -> drug binding site
- Build: Virtual screening pipeline — dock 1000 compounds to a GPCR using Boltz-2, rank by predicted affinity, compare to experimental IC50 values

## Interview Q&A — Membrane Protein Dynamics

**Q: Why are membrane proteins difficult to study experimentally and computationally?**
A: Experimental: membrane proteins require detergents/lipid nanodiscs for solubilization, low expression yields, resistant to crystallization (hence sparse PDB coverage). Computational: membrane environment is complex (asymmetric bilayer, lipid interactions), need specialized MD force fields (CHARMM36, MARTINI), and AF3 was trained mostly on soluble proteins.

**Q: What is the difference between TM topology prediction and 3D structure prediction?**
A: Topology prediction: which helices span the membrane (N-in/C-out orientation), 2D arrangement — fast, rule-based (Kyte-Doolittle hydrophobicity) or HMM (Phobius, TMHMM). 3D structure: full atomic coordinates — requires AF3, RoseTTAFold. Topology prediction is a prerequisite for understanding function even without a 3D structure.

**Q: Explain the GPCR conformational states and why they matter for drug design.**
A: GPCRs exist in inactive (ground state, binds antagonists/inverse agonists), active (bound to agonist + G-protein), and intermediate states. The active state is ~5Å different in TM6 outward displacement. Drug design: agonists stabilize active state; antagonists block activation; inverse agonists stabilize inactive. Knowing which state you're modeling is critical for virtual screening.

**Q: What is coarse-grained MD and when is it used instead of all-atom MD?**
A: CG-MD groups multiple atoms into "beads" (MARTINI: ~4 heavy atoms per bead). Time step: 20-40 fs vs 2 fs for all-atom. Access microsecond-to-millisecond timescales where GPCR conformational changes occur. Trade-off: loses atomic detail (can't model specific H-bonds, drug binding precisely). Used for: membrane self-assembly, lipid diffusion, protein-membrane interactions at long timescales.

### Membrane Protein — Time Complexity
| Operation | Time | Scale |
|-----------|------|-------|
| Kyte-Doolittle hydrophobicity scan | O(n) | microseconds |
| TMHMM/Phobius topology prediction | O(n) | seconds |
| ESM2 LoRA fine-tuning (N sequences) | O(N × L² × d) | hours on GPU |
| All-atom MD (1 ns, 50K atoms) | O(n_atoms × n_steps) | days on GPU |
| CG-MD (10 μs, 5K beads) | O(n_beads × n_steps) | days on GPU |
| Docking score regression | O(N × features) | minutes |
| Conformational VAE (N structures) | O(N × d) | hours |

In [ ]:
# Module 11 Resources
print("KEY RESOURCES — Module 11: Membrane Protein Dynamics")
print()
refs = {
    "Databases": [
        "MPstruc: membrane protein structures (blanco.biomol.uci.edu/mpstruc)",
        "OPM: orientations of proteins in membranes (opm.phar.umich.edu)",
        "STCRDab: T-cell receptor structures",
        "GPCRdb: GPCR structure + function database (gpcrdb.org)",
    ],
    "Tools": [
        "TMHMM: TM topology prediction (DTU Bioinformatics)",
        "DeepTMHMM: deep learning TM topology",
        "CHARMM-GUI: membrane builder for MD simulations",
        "MARTINI force field: cg.martini-go.org",
        "VMD: molecular visualization (ks.uiuc.edu/vmd)",
    ],
    "Papers": [
        "Lomize et al. 2022 — OPM database and PPM web server",
        "Jumper et al. 2021 — AF2 (some membrane protein benchmarks)",
        "Pfeiffenberger & Bates 2018 — Refinement of membrane proteins in AF",
    ],
}
for cat, items in refs.items():
    print(f"\n{cat}:")
    for item in items:
        print(f"  * {item}")

## Cheat Sheet — Membrane Protein ML Toolkit

### Vocabulary

| Term | Definition |
|------|------------|
| TM helix | Transmembrane alpha-helix; spans the lipid bilayer; ~20 hydrophobic residues |
| Topology | Inside/outside/TM labeling of each residue; fundamental structural property |
| GPCR | G protein-coupled receptor; 7-TM; largest drug target family; activates via G-proteins |
| Biased agonism | Drug activates G-protein OR beta-arrestin pathway (not both); pharmacological tool |
| OPM | Orientations of Proteins in Membranes; database giving bilayer position for PDB structures |
| MARTINI | Coarse-grained force field; ~4 atoms per bead; 100-1000x faster than all-atom MD |
| MSM | Markov State Model; discrete-state kinetic model built from MD trajectories |
| FEP | Free energy perturbation; alchemical method; gold-standard for binding affinity |
| NNP | Neural network potential; learned force field from QM or AA-MD data |
| ICL3 | Intracellular loop 3; disordered GPCR loop; couples to G-protein; often truncated in crystals |

### Key Numbers to Memorize

| Fact | Number |
|------|--------|
| Lipid bilayer thickness (hydrophobic core) | ~30 angstroms |
| Typical TM helix length | 20-25 residues |
| Human GPCR genes | ~800 |
| GPCR share of approved drugs | ~35% |
| Membrane protein share of human proteome | ~30% |
| Membrane protein share of PDB | ~10% |
| GPCR active-state TM6 displacement | ~14 angstroms |
| GPCR activation timescale | ~1 microsecond |
| MARTINI speedup vs all-atom MD | 100-1000x |
| OPM database size | ~9,000 structures |

### Fine-Tuning Decision Tree

```
Goal: Predict TM topology (inside/outside/TM per residue)
  -> ESM2 + LoRA (r=16) + 3-class token classification head
  -> Dataset: OPM 9k structures
  -> Eval: per-residue accuracy, segment-level topology accuracy

Goal: Predict GPCR conformational state (active/inactive)
  -> ESM2 embeddings + VAE or classifier
  -> Dataset: GPCRmd 160+ trajectories OR GPCRdb 2000+ structures
  -> Eval: state classification accuracy, latent space separation

Goal: Predict binding affinity for membrane protein-ligand
  -> Boltz-2 (fine-tune) OR ESM2 + GNN + delta-delta-G head
  -> Dataset: BindingDB membrane subset, ChEMBL GPCR assays
  -> Eval: Pearson r, RMSE in kcal/mol, virtual screening enrichment

Goal: Generate membrane protein conformational ensemble
  -> MDGen (diffusion) OR AlphaFlow (flow matching)
  -> Dataset: GPCRmd trajectories, MSM-sampled structures
  -> Eval: RMSD to true MD ensemble, Ramachandran plot quality
```

### Quick Reference: GPCR Classes

| Class | Example | Endogenous Ligand | Major Drug |
|-------|---------|------------------|------------|
| Class A (Rhodopsin) | beta2-adrenergic | Adrenaline | Metoprolol (heart) |
| Class A (Rhodopsin) | mu-opioid receptor | Endorphins | Morphine (pain) |
| Class A (Rhodopsin) | GLP-1 receptor | GLP-1 peptide | Semaglutide (Ozempic) |
| Class B (Secretin) | GLP-1 receptor | GLP-1 peptide | Exenatide (diabetes) |
| Class C (Glutamate) | mGluR5 | Glutamate | Mavoglurant (Fragile X) |
| Frizzled | Smoothened | Hedgehog pathway | Vismodegib (cancer) |

### Module Connections

```
03/01 (PDB, RMSD)  ----->  11/01: GPCR structure analysis, RMSD between active/inactive
06/01 (GNN)        ----->  11/01: EGNN for CG force field learning
07/01-04 (AF3)     ----->  11/01: Why AF3 fails for membrane proteins
10/01 (LoRA, ddG)  ----->  11/01: Apply LoRA to membrane proteins, ddG for variants
11/01 (this)       ----->  Original research / industry applications
```

## Mastery Check

On a first pass, success means you can:
1. explain why membrane proteins are a special ML problem
2. describe what topology prediction is doing
3. connect one ML choice to a membrane-specific biological constraint
4. decide whether to return later for deeper implementation work


---
## 🌐 Real GPCR Data from OPM Database

The Orientations of Proteins in Membranes (OPM) database contains >10,000 structures of membrane proteins with their calculated membrane insertion geometry. Here we fetch and analyze a real GPCR structure.


In [ ]:
# OPM DATABASE — Real GPCR membrane orientation data
# OPM: https://opm.phar.umich.edu/

import urllib.request
import json
import numpy as np
import matplotlib.pyplot as plt

def fetch_opm_protein(pdb_id):
    """Fetch membrane orientation data from OPM API."""
    url = f"https://opm.phar.umich.edu/api/proteins?search={pdb_id.upper()}"
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            data = json.loads(r.read().decode('utf-8'))
        return data
    except Exception as e:
        return None

# β2-adrenergic receptor (ADRB2) — the prototypical GPCR
# PDB: 2RH1 (bound to carazolol, a beta-blocker)
print("Fetching β2-Adrenergic Receptor (2RH1) from OPM...")
opm_data = fetch_opm_protein('2RH1')

if opm_data and 'objects' in str(opm_data):
    print(f"OPM data: {json.dumps(opm_data, indent=2)[:500]}...")
else:
    print("OPM API unavailable — using reference data")
    print()
    print("REFERENCE DATA for β2-Adrenergic Receptor (2RH1):")
    opm_reference = {
        "pdb_id": "2RH1",
        "name": "Beta-2 adrenergic receptor",
        "family": "Class A GPCR",
        "membrane_thickness": 30.0,  # Angstroms
        "tilt_angle": 7.0,           # Degrees from membrane normal
        "hydrophobic_thickness": 28.0,
        "n_transmembrane_helices": 7,
        "tm_segments": [
            (29, 60),    # TM1
            (67, 96),    # TM2
            (103, 129),  # TM3
            (147, 171),  # TM4
            (197, 225),  # TM5
            (267, 298),  # TM6
            (305, 328),  # TM7
        ],
        "ligand": "Carazolol (inverse agonist, beta-blocker)",
        "g_protein_coupling": "Gs (stimulatory)",
        "function": "Adrenaline/noradrenaline receptor; controls heart rate and bronchodilation",
    }

    for key, val in opm_reference.items():
        print(f"  {key}: {val}")

# Visualize TM helix topology (TM-topology cartoon)
fig, ax = plt.subplots(figsize=(12, 5))

tm_segments = [(29,60),(67,96),(103,129),(147,171),(197,225),(267,298),(305,328)]
seq_length = 365
total_residues = list(range(1, seq_length+1))

# Color by membrane location
membrane_start, membrane_end = 30, 60  # membrane z-coordinates (Å)

for i, (start, end) in enumerate(tm_segments):
    color = plt.cm.RdYlBu_r(i / len(tm_segments))
    ax.barh([0], [end-start], left=[start], height=0.8, color=color,
            alpha=0.85, label=f'TM{i+1}')

# Add loops
prev_end = 0
for i, (start, end) in enumerate(tm_segments):
    if prev_end > 0:
        ax.barh([0], [start-prev_end], left=[prev_end], height=0.3,
                color='lightgray', alpha=0.5)
    prev_end = end

ax.set_xlim(0, seq_length)
ax.set_ylim(-0.8, 1.2)
ax.set_xlabel('Residue number')
ax.set_title('β2-Adrenergic Receptor (2RH1) — Transmembrane Topology
'
             '7 transmembrane helices (blue → red), gray = loops')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.axvline(tm_segments[0][0], color='black', alpha=0.3, linestyle='--', label='TM boundaries')
ax.set_yticks([])

# Annotate key functional residues
key_residues = {
    'Asp113
(proton acceptor)': 113,
    'Asp79
(DRY motif)': 79,
    'Thr68
(Asn binding)': 68,
    'Lys305
(ligand)': 305,
}
for label, pos in key_residues.items():
    ax.axvline(pos, color='red', alpha=0.6, linewidth=1.5)
    ax.text(pos, 0.9, label, fontsize=7, ha='center', color='red', rotation=45)

plt.tight_layout()
plt.savefig('gpcr_topology.png', dpi=100)
plt.show()

print()
print("REAL TOOL FOR TM PREDICTION: DeepTMHMM")
print("  pip install pybiolib")
print("  biolib run DTU/DeepTMHMM --fasta my_protein.fasta")
print("  # Or use web server: https://dtu.biolib.com/DeepTMHMM")
print()
print("OUTPUT: Topology string like 'ooooMMMMMMMiiiiiMMMMMMMooo...'")
print("  'o' = outside membrane, 'i' = inside, 'M' = membrane")
print("  This topology string defines the graph structure for AF3 membrane protein prediction")

---
## 🎯 Capstone Mini-Project: GPCR Drug Binding Prediction

**Goal:** Train a simple classifier to predict whether a compound binds the β2-adrenergic receptor based on molecular features.

**Time:** ~3 hours | **Difficulty:** Intermediate

**What you'll practice:**
- Featurizing small molecules (Morgan fingerprints)
- Handling imbalanced datasets in drug discovery
- Interpreting a classifier for hit identification


In [ ]:
# CAPSTONE: GPCR LIGAND CLASSIFICATION
# Predict: does a molecule bind β2-adrenergic receptor?

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

np.random.seed(42)

# In production: load from ChEMBL or BindingDB
# ChEMBL has 2000+ ADRB2 ligands — pip install chembl_webresource_client
# Here we simulate with realistic molecular fingerprint statistics

n_actives = 200    # molecules that bind (IC50 < 1μM)
n_inactives = 800  # molecules that don't bind (random screening library)

# Simulate Morgan fingerprints (2048-bit)
# Actives have correlated features (chemical scaffold similarity)
from sklearn.datasets import make_classification
X_actives, _   = make_classification(n_samples=n_actives,  n_features=100,
                                      n_informative=20, n_redundant=5, random_state=1)
X_inactives, _ = make_classification(n_samples=n_inactives, n_features=100,
                                      n_informative=5,  n_redundant=2, random_state=2)

# Shift actives to simulate different chemical space
X_actives += np.random.normal(1.5, 0.5, X_actives.shape)

X = np.vstack([X_actives, X_inactives])
y = np.array([1]*n_actives + [0]*n_inactives)

print(f"Dataset: {n_actives} actives, {n_inactives} inactives ({n_actives/(n_actives+n_inactives)*100:.0f}% positive)")
print(f"Class ratio: 1:{n_inactives//n_actives} (imbalanced — typical of drug discovery)")

# Train with class balancing
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

aucs = cross_val_score(rf, X, y, cv=cv, scoring='roc_auc')
aps  = cross_val_score(rf, X, y, cv=cv, scoring='average_precision')

print(f"\n5-fold CV results:")
print(f"  ROC-AUC: {aucs.mean():.3f} ± {aucs.std():.3f}")
print(f"  Average Precision (PRC-AUC): {aps.mean():.3f} ± {aps.std():.3f}")
print()
print("METRIC CHOICE IN DRUG DISCOVERY:")
print("  ROC-AUC: measures overall ranking (ignores class imbalance)")
print("  PRC-AUC: measures precision on actives (better for imbalanced)")
print("  EF (Enrichment Factor): fraction of actives in top X% ranked → lab efficiency")
print()

# Enrichment Factor calculation (key metric in virtual screening)
rf.fit(X, y)
probs = rf.predict_proba(X)[:, 1]
sorted_idx = np.argsort(probs)[::-1]
top_10pct = sorted_idx[:len(X)//10]
ef10 = y[top_10pct].mean() / y.mean()

print(f"  Enrichment Factor (top 10%): {ef10:.2f}× random")
print(f"  → Screening top 10% of compounds finds {ef10:.1f}× more actives than random")
print()
print("CONNECT TO REAL DATA:")
print("  from chembl_webresource_client.new_client import new_client")
print("  activity = new_client.activity")
print("  res = activity.filter(target_chembl_id='CHEMBL210',  # ADRB2")
print("                        standard_type='IC50',")
print("                        standard_value__lte=1000).only(['molecule_chembl_id',")
print("                                                        'standard_value'])")

## 🐛 Debug Exercise — Membrane Protein Analysis

Find and fix the **3 bugs** in the GPCR/membrane protein analysis code below.

**Expected correct output:**
```
TM helices detected: 7  (GPCR has 7 transmembrane helices)
Membrane insertion depth: positive value measured from bilayer center
Helix residues: those with DSSP code 'H', not 'h' (case matters)
```

<details>
<summary>Show answer</summary>

**Bug 1 — Wrong hydrophobicity threshold:** A Kyte-Doolittle window score > 1.6 is the
standard threshold for TM helix prediction (not > 0.0). Using 0.0 finds far too many
"TM helices". Fix: `if window_score > 1.6`.

**Bug 2 — Membrane depth from wrong reference:** Membrane insertion depth should be
measured from the bilayer *center* (z=0 by convention), not from the outer leaflet surface.
Fix: `depth = abs(residue_z - bilayer_center_z)` where `bilayer_center_z = 0.0`.

**Bug 3 — DSSP helix code case-sensitive:** DSSP returns `'H'` for alpha-helix (uppercase).
Comparing with `'h'` (lowercase) finds zero helix residues.
Fix: `if dssp_code == 'H'` (uppercase).
</details>


In [ ]:
# DEBUG EXERCISE — Membrane protein analysis, find and fix 3 bugs
import numpy as np

# Kyte-Doolittle hydrophobicity scale
KD_SCALE = {
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2,
}

# Simulated GPCR-like sequence with 7 TM helices
# Real GPCR has alternating hydrophobic (TM) and hydrophilic (loop) stretches
np.random.seed(42)
TM_AAs  = 'ILMFVAW'   # hydrophobic
LOOP_AAs = 'RNDQEKHSP' # hydrophilic
tm_helix   = ''.join(np.random.choice(list(TM_AAs), 23))    # ~23 residue TM helix
loop_seg   = ''.join(np.random.choice(list(LOOP_AAs), 15))  # 15 residue loop
sequence   = (tm_helix + loop_seg) * 7 + tm_helix           # 7 TM + loops

def predict_tm_helices(seq, window=19):
    helices = []
    in_helix = False
    start = 0
    for i in range(len(seq) - window + 1):
        window_seq = seq[i:i+window]
        score = np.mean([KD_SCALE.get(aa, 0) for aa in window_seq])
        # BUG 1: threshold 0.0 is too low — finds hydrophobic loops too
        # Standard TM helix threshold (Kyte-Doolittle) is > 1.6
        if score > 0.0:     # should be > 1.6
            if not in_helix:
                start = i
                in_helix = True
        else:
            if in_helix:
                helices.append((start, i + window))
                in_helix = False
    return helices

helices = predict_tm_helices(sequence)
print(f"TM helices detected (threshold=0.0, too permissive): {len(helices)}")
print(f"Expected: 7 TM helices for a GPCR")

# BUG 2: membrane depth measured from outer leaflet (z=20) instead of bilayer center (z=0)
bilayer_half_width = 15.0   # angstroms
outer_leaflet_z = 20.0      # arbitrary
residue_z_coords = np.random.uniform(-bilayer_half_width, bilayer_half_width, 50)

# Should measure from bilayer center (z=0), not from outer leaflet
depths_wrong = np.abs(residue_z_coords - outer_leaflet_z)   # BUG: wrong reference plane
depths_correct = np.abs(residue_z_coords)                   # correct: from center
print(f"\nMean depth (wrong, from outer leaflet): {depths_wrong.mean():.2f} A")
print(f"Mean depth (correct, from bilayer center): {depths_correct.mean():.2f} A")

# BUG 3: DSSP alpha-helix code is uppercase 'H', not lowercase 'h'
dssp_codes = ['H', 'H', 'H', 'E', 'C', 'H', 'G', 'H']  # real DSSP output uses uppercase
helix_residues_wrong   = [i for i, c in enumerate(dssp_codes) if c == 'h']  # BUG
helix_residues_correct = [i for i, c in enumerate(dssp_codes) if c == 'H']
print(f"\nHelix residues (lowercase 'h' — wrong): {helix_residues_wrong}")
print(f"Helix residues (uppercase 'H' — correct): {helix_residues_correct}")
